In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("ForeignGifts_edu.csv")

In [4]:
uva = df[df["Institution Name"] == "University of Virginia"]

In [5]:
print(f"UVA total gifts: {len(uva)}")
print(f"UVA total amount: {uva['Foreign Gift Amount'].sum()}")

UVA total gifts: 98
UVA total amount: 22189238


In [6]:
gift_counts = df.groupby("Institution Name")["Foreign Gift Amount"].agg(["count", "sum",])

In [7]:
similar_by_count = gift_counts[(gift_counts["count"] >= 95) & (gift_counts["count"] <= 110)]
similar_by_count.sort_values("count", ascending=False)

,count,sum
Institution Name,,
Northwood University,105,11159755
"University of California, Santa Barbara",101,28640020
Purdue University,99,75504368
University of Virginia,98,22189238
Georgetown University,97,379950511


Realized this isn't really what part 2 is asking, so pivoting to analyzing schools based on looking from the outside - in. (Large Schools with roughly similar popularity)

In [8]:
uva_peers = [
    "University of Virginia",
    "University of Michigan - Ann Arbor",
    "University of California, Los Angeles",
    "University of California, Berkeley",
    "College of William & Mary",
    "Virginia Polytechnic Institute & State University",
    "Georgetown University",
]

peer_df = df[df["Institution Name"].isin(uva_peers)]
peer_df.groupby("Institution Name")["Foreign Gift Amount"].agg(["sum", "count", "mean", "median"]).sort_values("sum", ascending=False)

,sum,count,mean,median
Institution Name,,,,
Georgetown University,379950511,97,3.917016e+06,500000.0
"University of California, Berkeley",294479904,394,7.474109e+05,424025.0
University of Michigan - Ann Arbor,287336783,715,4.018696e+05,200000.0
"University of California, Los Angeles",242146363,3916,6.183513e+04,250.0
University of Virginia,22189238,98,2.264208e+05,80535.5
Virginia Polytechnic Institute & State University,11792062,24,4.913359e+05,350000.0
College of William & Mary,3384498,17,1.990881e+05,51600.0


In [9]:
total_market = df["Foreign Gift Amount"].sum()
uva = df[df["Institution Name"] == "University of Virginia"]
uva_total = uva["Foreign Gift Amount"].sum()

ranked = df.groupby("Institution Name")["Foreign Gift Amount"].sum().sort_values(ascending=False)
uva_rank = ranked.index.get_loc("University of Virginia") + 1

print(f"UVA total: ${uva_total:,.0f}")
print(f"National market total: ${total_market:,.0f}")
print(f"UVA share of market: {uva_total/total_market*100:.3f}%")
print(f"UVA rank: #{uva_rank} of {len(ranked)} institutions")

UVA total: $22,189,238
National market total: $16,600,515,514
UVA share of market: 0.134%
UVA rank: #101 of 318 institutions


In [10]:
uva_countries = set(uva["Country of Giftor"].unique())

top20_national = set(df.groupby("Country of Giftor")["Foreign Gift Amount"].sum().nlargest(20).index)

missing = top20_national - uva_countries
missing_ranked = (
    df[df["Country of Giftor"].isin(missing)]
    .groupby("Country of Giftor")["Foreign Gift Amount"]
    .sum()
    .sort_values(ascending=False)
)


print(missing_ranked)
print(top20_national)
print(uva_countries)

Country of Giftor
QATAR                   2706240869
CHINA                   1237952112
SAUDI ARABIA            1065205930
CANADA                   898160656
HONG KONG                887402529
INDIA                    539556490
GERMANY                  442475605
UNITED ARAB EMIRATES     431396357
AUSTRALIA                248409202
KUWAIT                   234106321
DENMARK                  173956102
BRAZIL                   158757039
Name: Foreign Gift Amount, dtype: int64
{'SINGAPORE', 'SAUDI ARABIA', 'AUSTRALIA', 'THE NETHERLANDS', 'BRAZIL', 'JAPAN', 'HONG KONG', 'DENMARK', 'FRANCE', 'SWEDEN', 'ENGLAND', 'UNITED ARAB EMIRATES', 'SWITZERLAND', 'KUWAIT', 'INDIA', 'QATAR', 'GERMANY', 'CHINA', 'CANADA', 'BERMUDA'}
{'SINGAPORE', 'ISRAEL', 'ENGLAND', 'BANGLADESH', 'BERMUDA', 'TANZANIA', 'FINLAND', 'GUERNSEY', 'FRANCE', 'SWEDEN', 'THE NETHERLANDS', 'SWITZERLAND', 'KOREA', 'RWANDA', 'JAPAN'}


In [11]:
for c in ["QATAR", "CHINA", "SAUDI ARABIA"]:
    n = df[df["Country of Giftor"] == c]["Institution Name"].nunique()
    print(f"{c}: gives to {n} different institutions")

QATAR: gives to 41 different institutions
CHINA: gives to 154 different institutions
SAUDI ARABIA: gives to 152 different institutions


Stategy for New Sources of Foreign Money: 

UVA ranks 101 out of 318 and behind peers like Georgetown, Michigan, and UCLA. Part of this gap comes because UVA is absent from many of the countries giving the most nationally. UVA has zero gifts from 12 of the top 20 national donor countries, including the three largest (Qatar, China, and Sauida Rabia). Going forward, UVA should look to target more of these top 20 countries, since they represent markets already proven to heavily fund U.S. Instituions. 